# 오토라벨링 수행 결과 보고서

## 1. 수행 개요
* **사용 모델**: `yolo_dataset/runs/yolov8_action_game/weights/best.pt`
* **추론 설정**: 기본 신뢰도 임계값(Confidence Threshold) `0.25` 적용
* **데이터 변환**:
  * 감지된 각 Bounding Box의 픽셀 좌표(`xyxy`)를 LabelMe 포인트 형식(`[[x_min, y_min], [x_max, y_max]]`)으로 변환 (`step1_convert.py`의 역변환 적용)
  * `shape_type`은 `"rectangle"` 지정
  * **클래스 매핑**:
    * `Class 0` $\rightarrow$ `user_character`
    * `Class 1` $\rightarrow$ `user_id`


## 2. 데이터 생성 결과
* **저장 위치**: `auto_labeled_data/`
* **생성 파일**: 총 922개 파일 (원본 이미지 파일 461개 + JSON 레이블 파일 461개)
  * JSON 파일 구조는 기존 `Labeled_data/` 폴더의 구조와 정확히 일치함 (버전 `"0.4.36"`, `imagePath`, `imageHeight`, `imageWidth` 등 포함, `imageData`는 `null`).
  * **AnyLabeling / LabelMe** 프로그램에서 해당 폴더를 즉시 열어 검토 가능.
* **감지된 객체 수**:
  * `user_character`: 160개
  * `user_id`: 99개

## 3. 분석 및 특이사항
* 전체 461개 이미지 중 **316개 이미지에서 객체가 전혀 감지되지 않음** (해당 JSON 내 도형 목록이 비어 있으나 파일 포맷은 정상적이고 유효함).
* 이는 약 **69%의 미탐지율(오탐률)**을 의미하며 다음 원인들을 고려할 수 있음:
  1. 모델 성능 문제로 인해 실제로 존재하는 객체를 감지하지 못함
  2. 많은 스크린샷 원본 이미지에 해당 캐릭터/ID가 애초에 포함되어 있지 않음
  3. 새롭게 추론한 이미지들이 기존 학습 데이터의 분포와 다름

<div style="display: flex; justify-content: space-between; text-align: center;">
  <div style="width: 48%;">
    <img src="auto_label_005.PNG"><br>
    <i>사진 1. user_character 탐지 및 미탐/오탐 사례</i>
  </div>

  <div style="width: 48%;">
    <img src="auto_label_004.PNG"><br>
    <i>사진 2. user_id 오탐 사례</i>
  </div>
</div>

<br>

#### **사진 1: user_character 관련 성공 및 실패**
**성공적인 탐지**: 화려한 스킬 이펙트나 가림(Occlusion) 속에서도 캐릭터의 남은 윤곽선과 패턴을 통해 `user_character`를 올바르게 탐지해냅니다.

**오탐 및 미탐 사례**: 주변 배경의 사물(예: 기둥)이나 특정 오브젝트 픽셀 패턴이 캐릭터와 수학적으로 유사할 경우 잘못 탐지(False Positive)할 수 있으며, 이펙트에 의해 캐릭터 특징을 완전히 상실한 경우 놓치는 미탐(False Negative)이 발생합니다.

#### **사진 2: user_id 오탐지 원인 파악**
**상황**: `user_character`는 정상적으로 탐지했으나, 전혀 다른 UI 텍스트(예: 데미지 수치, 스킬명 등)를 `user_id`로 오탐지했습니다.

**왜 이런 오탐이 발생했는가?**
1. **객체 탐지 모델의 본질적 한계(No OCR)**: YOLO는 텍스트의 '의미(문자 내용)'를 읽는 모델이 아닙니다. "작고 가로로 긴 픽셀 덩어리"라는 시각적 패턴만을 인식합니다.
2. **형태적 유사성(Feature Confusion)**: 액션 게임에서는 수많은 데미지 숫자, 콤보 텍스트, 스킬명, NPC 이름 등이 화면에 나타납니다. 이들 역시 작고 가로로 긴 텍스트 형태를 가지므로, 모델 입장에서는 학습 데이터에서 본 `user_id`의 형태적 특징과 사실상 동일하다고 판단해버립니다. (이를 '문맥적 깊이의 오류'로 해석하는 것은 잘못되었으며, 단순히 형태학적인 시각 특징 혼동입니다.)

## 4. 향후 검토 및 개선 권장사항
* 검토 시 모델이 실제 객체를 많이 놓친 것(미탐)으로 판단된다면, **임계값을 낮추어 재실행**해 볼 것을 권장합니다.
  * **코드 변경 예시**: `auto_label_images(..., conf_threshold=0.1)`
* 어차피 수동으로 검토/수정하는 과정이 따르므로, 모델이 예측한 박스가 너무 안 나오는 것보다는 박스가 다소 많게 나오더라도(오탐 증가) 미탐을 줄이는 것이 라벨링 작업 시 직접 박스를 그리는 수고를 덜 수 있어 일반적으로 더 유리합니다.

---
## 6. 오탐 분석을 통한 현재 한계점 및 개선 방향

위의 사진 분석을 종합해 볼 때, 현재 오토라벨링 파이프라인이 가지는 한계점과 이를 해결하기 위한 개선 방향은 다음과 같습니다.

### 현재 파이프라인의 한계점
1. **텍스트 의미 기반의 분류 불가**: 객체 탐지 모델(YOLO)은 텍스트의 내용(데미지 숫자, 스킬명 vs 실제 유저 닉네임)을 읽지 못합니다. 형태적인 픽셀 패턴에만 의존하므로 잦은 오탐이 발생합니다.
2. **공간적/논리적 문맥 파악 부재**: `user_id`는 논리적으로 항상 `user_character`의 근처(주로 위쪽)에 위치해야 하지만, 현재 모델은 이러한 '공간적 관계성'을 고려하지 않고 화면 내 어디서든 비슷한 텍스트 덩어리가 보이면 단독으로 `user_id`라고 탐지해버립니다.
3. **데이터셋 내 Negative 샘플(오답 데이터) 부족**: 데미지 이펙트나 일반 UI 텍스트와 같이 모델이 헷갈리기 쉬운 요소들을 '정답이 아님'으로 학습할 만한 충분한 다양성의 데이터가 없었거나 클래스 세분화가 이루어지지 않았습니다.

### 개선 방향
1. **Post-processing(후처리) 로직 추가 - 공간 휴리스틱(Spatial Heuristics)**:
   - 추론 결과 단계에서 추가적인 파이썬 스크립트 로직을 도입합니다.
   - 예측된 `user_id` 박스를 검사하여, 해당 좌표 아래에 가까운 거리 내에 `user_character` 박스가 존재하지 않는다면, 이는 데미지 수치나 일반 텍스트로 간주하고 **자동으로 삭제**하는 필터링 로직을 추가할 수 있습니다.
2. **OCR 파이프라인 결합 (Hybrid Approach)**:
   - YOLO가 `user_id` 영역을 탐지하면, 해당 박스를 잘라내어(Crop) 가벼운 광학 문자 인식 엔진(OCR, 예: Tesseract, EasyOCR 등)에 넘깁니다.
   - 인식된 텍스트가 순수 숫자(데미지)이거나 특정 스킬명일 경우 오답으로 판별하여 결과를 삭제하게 설계하면 정확도가 극대화됩니다.
3. **데이터셋 확장 및 클래스 세분화 (Data-centric Approach)**:
   - 기존 라벨링 데이터에 `damage_number`, `npc_name`, `system_text` 등 헷갈리기 쉬운 텍스트를 새로운 클래스로 명시적으로 라벨링하여 추가 학습시킵니다.
   - 모델 스스로 색상, 테두리 패턴, 폰트의 미세한 픽셀 차이를 구별하는 법을 학습하게 됩니다.

---
## Additional. 작동 메커니즘 상세

`step6_auto_label.py` 코드는 라벨링되지 않은 원본 이미지들을 불러와 학습된 YOLOv8 모델로 추론을 수행하고, 그 결과를 LabelMe 호환 포맷(AnyLabeling 사용 가능)으로 자동 변환하여 저장하는 역할을 합니다. 구체적인 동작 메커니즘은 다음과 같습니다.

### 1) 환경 설정 및 사전 검증
- `ultralytics` 패키지의 `YOLO` 객체를 사용합니다.
- `unlabeled_dir`(입력 폴더)와 `weights_path`(모델 가중치, `best.pt`)의 존재 여부를 확인합니다.
- `output_dir`(`auto_labeled_data/`) 폴더를 생성합니다. 폴더가 이미 있다면 오류 없이 덮어쓸 수 있게 준비합니다(`exist_ok=True`).
- 확장자가 `.jpg, .jpeg, .png, .bmp`인 이미지 파일만 필터링하여 리스트업합니다.

### 2) 모델 로드 및 추론 수행
- `YOLO(weights_path)`를 호출하여 미리 학습한 YOLOv8 모델을 메모리에 로드합니다.
- 이미지 리스트를 순회하며 `model(image_path, conf=conf_threshold, verbose=False)`를 호출해 추론을 진행합니다.
  - `conf_threshold`는 기본값 `0.25`로 설정되어 있으며, 이 수치 이상의 신뢰도를 가진 객체만 예측값으로 반환됩니다.

### 3) 좌표 변환 및 포맷팅 (YOLO $\rightarrow$ LabelMe)
- 추론 결과물(`result.boxes`)을 순회하며 다음 작업을 수행합니다:
  1. **클래스 매핑**: YOLO의 숫자 클래스 ID(`0, 1`)를 LabelMe용 문자열 텍스트(`0: "user_character", 1: "user_id"`)로 변환합니다.
  2. **좌표 추출**: 예측된 Bounding Box 좌표(`xyxy` 포맷: `x_min, y_min, x_max, y_max`)를 추출합니다.
  3. **좌표 클램핑(Clamping)**: 혹시 모델이 이미지 크기를 벗어난 좌표를 예측했을 경우를 대비해, 좌표값을 `0`부터 `이미지 가로/세로 길이` 사이로 제한(`max`, `min` 함수 사용)하여 오류를 방지합니다.
  4. **Shape 구성**: LabelMe JSON 형식의 `shapes` 리스트에 들어갈 딕셔너리를 구성합니다.
     - `shape_type`: `"rectangle"`로 고정
     - `points`: `[[x_min, y_min], [x_max, y_max]]` 형태로 저장

### 4) JSON 파일 저장 및 원본 이미지 복사
- 각 이미지의 원본 해상도(`img_height`, `img_width`)를 바탕으로 LabelMe의 기본 JSON 포맷을 조립합니다.
  - `version: "0.4.36"`, `imagePath: 이미지 파일명`, `imageData: None`
- 조립된 딕셔너리(`labelme_data`)를 `json.dump`를 사용해 이미지와 동일한 이름의 `.json` 파일로 저장합니다.
- 마지막으로 `shutil.copy2`를 이용해 **원본 이미지 파일도 JSON 파일이 저장된 위치로 함께 복사**합니다.
  - 이는 LabelMe 또는 AnyLabeling에서 해당 디렉토리를 열었을 때 이미지와 라벨링 정보가 쌍(Pair)으로 즉시 매칭되어 시각적으로 확인 및 수정이 가능하도록 하기 위함입니다.

### 5) 처리 결과 요약 출력
- 반복문 종료 후, 성공적으로 라벨링된 전체 이미지 수와 **객체가 하나도 검출되지 않은 이미지 수(empty_count)** 를 터미널에 출력하여 작업자가 바로 결과를 확인할 수 있도록 돕습니다.